In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

In [2]:
products = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

In [8]:
@tool
def get_product(name: str) -> str:
    """returns the product details"""
    product = products.get(name.lower())
    if not product:
        return "product not found"
    return product

In [9]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.0,
    reasoning_format="parsed",
)

agent = create_agent(
    llm, 
    tools=[get_product], 
    system_prompt="you are a helpful assistant",
)

In [16]:
def ask(question: str) -> None:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [11]:
ask("What is price of smart watch?")

The smart watch is priced at **$199.99**.


In [12]:
reviews = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}

In [13]:
@tool
def get_reviews(name: str) -> str:
    """Get the reviews for a given name."""
    review = reviews.get(name)
    if not review:
        return "no reviews found for {name}"
    return review

In [14]:
agent = create_agent(
    llm, 
    tools=[get_product, get_reviews], 
    system_prompt="you are a helpful assistant",
)

In [18]:
ask("what is price and reviews of smart watch?")

**Smart Watch**

- **Price:** $199.99  
- **Reviews:** 340 reviews (average rating 3.9/5)


In [19]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    llm, 
    tools=[get_product, get_reviews], 
    system_prompt="You are a helpful assistant that helps users find the best product reviews.",
    checkpointer=InMemorySaver(),
)

In [22]:
def ask(question):
    configuration = {"configurable": {"thread_id": "session-1"}}
    response = agent.invoke({"messages": [{"role": "user", "content": question}]}, config=configuration)
    print(response["messages"][-1].content)

In [23]:
ask("What is the price of wireless headphones?")

The wireless headphones are priced at **$79.99**.


In [24]:
ask("What is its rating?")

The wireless headphones have a rating of **4.6 out of 5**.
